<a href="https://colab.research.google.com/github/treborskrub/Modular-/blob/main/audit_processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any

@dataclass
class Step:
    """Single auditable step in the pipeline."""
    step_id: int
    confidence: float
    grounding: float
    note: str = ""

@dataclass
class Result:
    """Audit result with full contraction history."""
    round_id: int
    step_id: int
    original_confidence: float
    original_grounding: float
    adjusted_confidence: float
    adjusted_grounding: float
    shortfall: float
    lambda_ratio: float
    gate_fired: bool
    verdict: str
    correction: float
    note: str

class UniversalProcessEngine:
    """Universal recursive contraction + audit engine."""
    def __init__(self, threshold: float = 1/3, anomaly_floor: float = 0.0001) -> None:
        self.threshold = threshold
        self.anomaly_floor = anomaly_floor
        self.initial_shortfall: Optional[float] = None
        self.history: List[Result] = []

    def apply_anomaly_floor(self, confidence: float, grounding: float) -> tuple[float, float]:
        confidence = max(0.0, confidence - self.anomaly_floor)
        grounding = max(0.0, grounding + self.anomaly_floor)
        return confidence, grounding

    def compute_shortfall(self, confidence: float, grounding: float) -> float:
        return max(0.0, confidence - grounding)

    def compute_lambda(self, shortfall: float) -> float:
        if self.initial_shortfall is None or self.initial_shortfall == 0:
            return 1.0
        return shortfall / self.initial_shortfall

    def classify(self, shortfall: float, lambda_ratio: float) -> str:
        if shortfall <= self.threshold:
            return "PASS"
        if lambda_ratio < 1.0:
            return "WARN"
        return "FAIL"

    def correction(self, shortfall: float, lambda_ratio: float) -> float:
        if shortfall <= self.threshold:
            return 0.0
        base = shortfall - self.threshold
        if lambda_ratio < 1.0:
            return min(1.0, base * 1.5)
        return min(1.0, base + self.anomaly_floor)

    def revise_step(self, step: Step, corr: float) -> Step:
        new_conf = max(0.0, step.confidence - corr * 0.4)
        new_ground = min(1.0, step.grounding + corr * 0.6)
        return Step(
            step_id=step.step_id,
            confidence=new_conf,
            grounding=new_ground,
            note=step.note + " | revised"
        )

    def audit_step(self, step: Step, round_id: int) -> Result:
        adj_conf, adj_ground = self.apply_anomaly_floor(step.confidence, step.grounding)
        shortfall = self.compute_shortfall(adj_conf, adj_ground)

        if self.initial_shortfall is None:
            self.initial_shortfall = shortfall

        lambda_ratio = self.compute_lambda(shortfall)
        gate_fired = shortfall > self.threshold
        verdict = self.classify(shortfall, lambda_ratio)
        corr = self.correction(shortfall, lambda_ratio)

        result = Result(
            round_id=round_id, step_id=step.step_id,
            original_confidence=step.confidence, original_grounding=step.grounding,
            adjusted_confidence=adj_conf, adjusted_grounding=adj_ground,
            shortfall=shortfall, lambda_ratio=lambda_ratio, gate_fired=gate_fired,
            verdict=verdict, correction=corr, note=step.note
        )
        self.history.append(result)
        return result

    def run(self, timeline: List[Step], max_rounds: int = 5) -> List[Result]:
        current = timeline
        for round_id in range(1, max_rounds + 1):
            next_timeline: List[Step] = []
            any_fail = False
            for step in current:
                result = self.audit_step(step, round_id)
                if result.verdict == "FAIL":
                    any_fail = True
                next_timeline.append(self.revise_step(step, result.correction))
            if not any_fail:
                break
            current = next_timeline
        return self.history

    def export_json(self) -> str:
        """Return compact JSON-ready string of all results."""
        return str([asdict(r) for r in self.history])